In [ ]:
# 식품 영양 정보를 가져와서 csv파일로 저장
# 가져올 정보. 식품명, 탄수화물, 지방, 단백질, 나트륨, 총칼로리
# 내 인증키: 34cf2a75ed8a3fd4404f4da20bb50ee752fd728f079f1dcc0cee0813fe3a649d

In [5]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

def get_food_nutrition_xml(service_key: str, food_name: str = ''):
  
    url = 'https://apis.data.go.kr/1471000/FoodNtrCpntDbInfo02/getFoodNtrCpntDbInq02'
    
    params = {
        'serviceKey': service_key,
        'FOOD_NM_KR': food_name,
        'pageNo': '1',
        'numOfRows': '100'
    }
    
    try:
        print(f"[{food_name if food_name else '전체'}] 데이터를 수집 중...")
        response = requests.get(url, params=params)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'xml')
        items = soup.find_all('item')
        
        if not items:
            print("검색 결과가 없습니다.")
            return None
            
        food_list = []
        for item in items:
            def get_val(tag_name):
                tag = item.find(tag_name)
                return tag.text if tag and tag.text else "0"

            food_list.append({
                '식품명': get_val('FOOD_NM_KR'),
                '탄수화물(g)': get_val('AMT_NUM2'),
                '단백질(g)': get_val('AMT_NUM3'),
                '지방(g)': get_val('AMT_NUM4'),
                '나트륨(mg)': get_val('AMT_NUM13'),
                '총칼로리(kcal)': get_val('AMT_NUM1'),
            })
            
        return pd.DataFrame(food_list)

    except Exception as e:
        print(f"오류 발생: {e}")
        return None

def save_to_csv(df, keyword):
    if df is not None:
        file_name = f'food_data_{keyword}.csv'
        df.to_csv(file_name, index=False, encoding='utf-8-sig')
        print(f"저장이 완료되었습니다: {file_name} (총 {len(df)}건)")

if __name__ == "__main__":
    MY_KEY = '34cf2a75ed8a3fd4404f4da20bb50ee752fd728f079f1dcc0cee0813fe3a649d'
    keyword = input("검색할 식품명: ")
    
    result_df = get_food_nutrition_xml(MY_KEY, keyword)
    
    if result_df is not None:
        print("\n--- 데이터 미리보기 ---")
        print(result_df.head())
        save_to_csv(result_df, keyword)

[치킨] 데이터를 수집 중...

--- 데이터 미리보기 ---
                 식품명 탄수화물(g) 단백질(g)  지방(g)  나트륨(mg) 총칼로리(kcal)
0             치킨데리야끼   58.00  16.54   9.01  267.000    208.000
1          무초절임(치킨무)   95.40   0.52   0.30  189.000      17.00
2  리소토/리조또_치킨 크림 리조또       0   6.22      0  217.000    185.000
3        도넛_올드훼션드먼치킨       0   3.57  25.71  214.000    471.000
4     도넛_레드벨벳 먼치킨 도넛       0   3.57  27.14  193.000    486.000
저장이 완료되었습니다: food_data_치킨.csv (총 100건)
